# Build Metabolic Model (Advanced)

To build a metabolic model we need the following inputs:
 - template (genome scale)
 - template (core)
 - atp metabolism medias - ModelSEEDpy provides an array of atp metabolism medias
 - genome
 - genome growth medium

In [12]:
import modelseedpy
from modelseedpy.core.mstemplate import MSTemplateBuilder
from modelseedpy import MSGenome

## Imports

- `modelseedpy`: The main ModelSEEDpy package for metabolic model reconstruction
- `MSTemplateBuilder`: Loads and constructs biochemical reaction templates used to populate models
- `MSGenome`: Parses genome protein FASTA files and stores annotated gene features

### Load Templates

In [44]:
def example_load_teamplates_from_file():
    import json
    with open('./Core-V5.2.json') as fh:
        template_core = MSTemplateBuilder.from_dict(json.load(fh)).build()
    with open('./GramNegModelTemplateV6.json') as fh:
        template_gramneg = MSTemplateBuilder.from_dict(json.load(fh)).build()
    return template_core, template_gramneg
def example_load_teamplates_from_url():
    import requests
    url = 'https://raw.githubusercontent.com/ModelSEED/ModelSEEDTemplates/refs/heads/main/templates/v7.0/Core-V6.json'
    from modelseedpy.core.mstemplate import MSTemplateBuilder
    template_core = MSTemplateBuilder.from_dict(requests.get(url).json()).build()
    url = 'https://raw.githubusercontent.com/ModelSEED/ModelSEEDTemplates/refs/heads/main/templates/v7.0/GramNegModelTemplateV7.json'
    template_gramneg = MSTemplateBuilder.from_dict(requests.get(url).json()).build()
    return template_core, template_gramneg

In [45]:
template_core, template_gramneg = example_load_teamplates_from_url()

Two templates are required for the reconstruction pipeline:

- **Core template** (`template_core`): Contains core metabolic reactions (e.g., central carbon metabolism, ATP synthesis). Used during the ATP correction step to detect thermodynamically infeasible cycles.
- **Gram-negative template** (`template_gramneg`): A genome-scale template tailored for gram-negative bacteria. Defines the full biochemical reaction space for model reconstruction and gapfilling.

Inspect each template object in the cells below to see the number of reactions, compounds, and compartments included.

In [10]:
template_core

ID,CoreTemplateBacteria
Memory address,0x07d2a755d7070
Number of metabolites,197
Number of species,232
Number of reactions,252
Number of biomasses,0
Number of roles,706
Number of complexes,468


In [11]:
template_gramneg

ID,GramNegative.modeltemplate
Memory address,0x07d2a75648a00
Number of metabolites,6573
Number of species,7116
Number of reactions,8584
Number of biomasses,1
Number of roles,20548
Number of complexes,3296


In [22]:
from pathlib import Path
path = Path('~/ADP1/data/ref_genome/GCF_000046845.1_ASM4684v1_protein.faa.gz')
path.expanduser().resolve()

PosixPath('/storage/fliu/data/ADP1/ref_genome/GCF_000046845.1_ASM4684v1_protein.faa.gz')

The genome input is a **protein FASTA file** (`.faa`) containing the amino acid sequences of all predicted protein-coding genes. Here, the genome for *Acinetobacter baylyi* ADP1 (`GCF_000046845.1`) is used as an example. Verify the path resolves correctly before loading.

### Load Genome and Annotate with RAST

In [23]:
#genome = modelseedpy.MSGenome.from_fasta('/opt/python/ModelSEEDpy/examples/Model Reconstruction/GCF_000005845.2_ASM584v2_protein.faa')
path = '/storage/fliu/data/ADP1/ref_genome/GCF_000046845.1_ASM4684v1_protein.faa.gz'
genome = MSGenome.from_fasta(path)

In [25]:
rast = modelseedpy.RastClient()
out = rast.annotate_genome(genome)

**RAST** (Rapid Annotation using Subsystem Technology) assigns functional roles and ontology terms to each protein-coding feature. These annotations map genome features to reactions in the template, forming the biochemical basis of the metabolic model.

In [26]:
for feature in genome.features:
    print(feature.ontology_terms)

{'RAST': ['LSU ribosomal protein L28p', ' LSU ribosomal protein L28p, zinc-independent']}
{'RAST': ['Two-component transcriptional response regulator, OmpR family']}
{'RAST': ['LSU ribosomal protein L20p']}
{'RAST': ['Cytochrome d ubiquinol oxidase subunit X (EC 1.10.3.-)']}
{'RAST': ['ATP synthase F0 sector subunit c (EC 3.6.3.14)']}
{'RAST': ['LSU ribosomal protein L34p']}
{'RAST': ['LSU ribosomal protein L36p', ' LSU ribosomal protein L36p, zinc-dependent']}
{'RAST': ['SSU ribosomal protein S11p (S14e)']}
{'RAST': ['DNA-binding protein Fis']}
{'RAST': ['SSU ribosomal protein S21p']}
{'RAST': ['LSU ribosomal protein L33p', ' LSU ribosomal protein L33p, zinc-independent']}
{'RAST': ['Glycyl-tRNA synthetase alpha chain (EC 6.1.1.14)']}
{'RAST': ['SSU ribosomal protein S9p (S16e)']}
{'RAST': ['SSU ribosomal protein S12p (S23e)']}
{'RAST': ['LSU ribosomal protein L35p']}
{'RAST': ['LSU ribosomal protein L13p (L13Ae)']}
{'RAST': ['4Fe-4S dicluster fused with DUF3470']}
{'RAST': ['Nitrogen

### 1 - Build Base Model

In [27]:
builder = modelseedpy.MSBuilder(genome, template_gramneg, 'test')

In [28]:
model_base = builder.build_base_model('test', annotate_with_rast=False)

`build_base_model` maps annotated genome features to template reactions via their functional role assignments. The resulting **base model** contains only reactions with direct gene evidence. At this stage the model is typically incomplete — it lacks reactions needed to carry biomass flux, which will be added during gapfilling.

In [29]:
builder.add_atpm(model_base)
model_base

Name,test
Memory address,7d2a6caa4160
Number of metabolites,1188
Number of reactions,1286
Number of genes,915
Number of groups,674
Objective expression,1.0*bio1 - 1.0*bio1_reverse_b18f7
Compartments,"Cytosol, Extracellular"


An **ATP maintenance (ATPM) reaction** is added to account for non-growth-associated ATP consumption. This reaction is also used as the objective during the ATP correction step to detect reactions that generate ATP without thermodynamic justification.

### 2 - ATP Gapfill / Analysis

In [30]:
from modelseedpy.core.msatpcorrection import load_default_medias

In [31]:
default_medias = load_default_medias()
print(f'loaded {len(default_medias)} medias')

loaded 54 medias


`load_default_medias()` returns a list of `(MSMedia, min_objective_value)` tuples covering a range of metabolic environments. During ATP correction the model is tested on each medium to find the minimal set of reactions that prevent unrealistic ATP production across all conditions.

In [32]:
for media, min_obj in default_medias:
    print(media.id, media.get_media_constraints())
    break

Glc.O2 {'cpd00027_e0': (-1, 1000), 'cpd00007_e0': (-1000, 1000), 'cpd00001_e0': (-1000, 1000), 'cpd00067_e0': (-1000, 1000)}


In [33]:
from modelseedpy import MSATPCorrection
atp_correction = MSATPCorrection(model_base, template_core, default_medias,
                                 compartment='c0', atp_hydrolysis_id='ATPM_c0', 
                                 load_default_medias=False)

`MSATPCorrection` identifies and corrects reactions that create thermodynamically infeasible ATP-generating cycles. It uses the **core template** to propose minimal gapfilling solutions that constrain ATP flux to biologically realistic levels.

Key parameters:
- `model_base` — the draft model to correct
- `template_core` — source of corrective reactions
- `default_medias` — list of `(media, threshold)` conditions to test
- `compartment` / `atp_hydrolysis_id` — locates the ATPM reaction in the model

In [34]:
media_eval = atp_correction.evaluate_growth_media()
atp_correction.determine_growth_media()
atp_correction.apply_growth_media_gapfilling()
atp_correction.expand_model_to_genome_scale()
tests = atp_correction.build_tests()

INFO:modelseedpy.core.msmodelutl:cpd08021 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00811 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd08021 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00811 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd11632 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd08701 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd01024 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd01024 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00187 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00425 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00441 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd11632 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd08021 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00811 not found in model!
INFO:modelseedpy.core.msmodelutl:Expansion time:empty:0.5520334476605058
INFO:modelseedpy.core.msmodelutl:Filtered count:2 out of 14

In [35]:
for t in tests:
    print(t['media'].id, t['threshold'], atp_correction.media_gapfill_stats[t['media']])

empty 1e-05 None
Ac.O2 8.999999999989768 {'reversed': {}, 'new': {}}
Etho.O2 14.7 {'reversed': {}, 'new': {}}
Pyr.O2 12.45 {'reversed': {}, 'new': {}}
Fum.O2 16.2 {'reversed': {}, 'new': {}}
LLac.O2 14.7 {'reversed': {}, 'new': {}}
Dlac.O2 14.7 {'reversed': {}, 'new': {}}
For.O2 2.25 {'reversed': {}, 'new': {}}
Pyr 0.6 {'reversed': {}, 'new': {}}
Fum 1.5 {'reversed': {}, 'new': {}}
For.NO2 0.1000000000001819 {'reversed': {}, 'new': {}}
Pyr.NO3 4.950000000000001 {'reversed': {}, 'new': {}}
Ac.NO2 0.39999999999990904 {'reversed': {}, 'new': {}}
Ac.NO3 3.0 {'reversed': {}, 'new': {}}
Pyr.DMSO 0.6 {'reversed': {}, 'new': {}}
Pyr.TMAO 0.6 {'reversed': {}, 'new': {}}
Pyr.SO3 0.6 {'reversed': {}, 'new': {}}


Each test specifies a medium and a maximum allowed ATPM flux (`threshold`). The `media_gapfill_stats` column reports the number of reactions added per medium during gapfilling. Once these tests pass, the ATP-corrected model is ready for genome-scale gapfilling.

### 3 - Genome Scale Gapfilling

In [36]:
from modelseedpy import MSMedia
media_glucose = MSMedia.from_dict({
    'cpd00149': (-100, 100),
    'cpd00099': (-100, 100),
    'cpd00067': (-100, 100),
    'cpd00063': (-100, 100),
 'cpd00058': (-100, 100),
 'cpd00048': (-100, 100),
 'cpd00034': (-100, 100),
 'cpd00030': (-100, 100),
 'cpd00013': (-100, 100),
 'cpd00009': (-100, 100),
 'cpd00001': (-100, 100),
 'cpd00007': (-10, 100),
 'cpd00205': (-100, 100),
 'cpd00254': (-100, 100),
 'cpd00971': (-100, 100),
 'cpd10515': (-100, 100),
 'cpd10516': (-100, 100),
 'cpd11574': (-100, 100),
 'cpd00244': (-100, 100),
 'cpd00027': (-5, 100)})

The growth medium is defined as a dictionary mapping **ModelSEED compound IDs** to `(lower_bound, upper_bound)` flux tuples. A negative lower bound allows uptake; a positive upper bound allows secretion. Here, glucose (`cpd00027`) is the sole carbon source with an uptake limit of 5 mmol/gDW/h. All other compounds are minerals, ions, and water available in excess.

In [37]:
from modelseedpy import MSGapfill
gapfill = MSGapfill(model_base, default_gapfill_templates=[template_gramneg], test_conditions=tests, default_target='bio1')

`MSGapfill` searches the template for the minimum set of reactions to add so that the model can carry flux through the target reaction (biomass, `bio1`). Passing `test_conditions` ensures that gapfilling solutions are checked against the ATP correction tests and do not reintroduce thermodynamically infeasible cycles.

In [38]:
gapfill_res = gapfill.run_gapfilling(media_glucose)

INFO:modelseedpy.core.msmodelutl:Expansion time:empty:149.256893848069
INFO:modelseedpy.core.msmodelutl:Filtered count:237 out of 13012
INFO:modelseedpy.core.msmodelutl:Expansion time:Ac.O2:49.21201362926513
INFO:modelseedpy.core.msmodelutl:Filtered count:266 out of 13012
INFO:modelseedpy.core.msmodelutl:Expansion time:Etho.O2:0.39810003619641066
INFO:modelseedpy.core.msmodelutl:Filtered count:266 out of 13012
INFO:modelseedpy.core.msmodelutl:Expansion time:Pyr.O2:0.2605934329330921
INFO:modelseedpy.core.msmodelutl:Filtered count:266 out of 13012
INFO:modelseedpy.core.msmodelutl:Expansion time:Fum.O2:0.26289962232112885
INFO:modelseedpy.core.msmodelutl:Filtered count:266 out of 13012
INFO:modelseedpy.core.msmodelutl:Expansion time:LLac.O2:0.26514785271137953
INFO:modelseedpy.core.msmodelutl:Filtered count:266 out of 13012
INFO:modelseedpy.core.msmodelutl:Expansion time:Dlac.O2:0.26177208311855793
INFO:modelseedpy.core.msmodelutl:Filtered count:266 out of 13012
INFO:modelseedpy.core.msm

In [39]:
gapfill_res

{'reversed': {},
 'new': {'EX_cpd00030_e0': '<',
  'EX_cpd00063_e0': '<',
  'EX_cpd00027_e0': '>',
  'EX_cpd00099_e0': '<',
  'EX_cpd15378_e0': '>',
  'rxn12008_c0': '<',
  'rxn08333_c0': '>',
  'rxn02898_c0': '>',
  'rxn05459_c0': '>',
  'rxn05514_c0': '<',
  'rxn02160_c0': '>',
  'rxn00340_c0': '>',
  'rxn05039_c0': '>',
  'rxn03891_c0': '>',
  'rxn09997_c0': '<',
  'rxn09042_c0': '>',
  'rxn03536_c0': '>',
  'rxn05573_c0': '>',
  'rxn00856_c0': '>',
  'rxn00003_c0': '<',
  'rxn03537_c0': '>',
  'rxn12376_c0': '<',
  'rxn04675_c0': '>',
  'rxn06023_c0': '>',
  'rxn13906_c0': '<',
  'rxn01539_c0': '<',
  'rxn10094_c0': '>',
  'rxn02155_c0': '>',
  'rxn03150_c0': '>',
  'rxn03538_c0': '>',
  'rxn09310_c0': '>',
  'rxn01213_c0': '>',
  'rxn02304_c0': '>',
  'rxn05618_c0': '>',
  'rxn03393_c0': '>',
  'rxn00599_c0': '>',
  'rxn09040_c0': '>',
  'rxn00551_c0': '>',
  'rxn00216_c0': '>',
  'rxn01790_c0': '<',
  'rxn02897_c0': '>',
  'rxn05144_c0': '>',
  'rxn02831_c0': '>',
  'rxn02832_c0'

`gapfill_res` is a dictionary with two keys:
- `'new'`: reactions to add, mapped to their direction (`'>'` forward, `'<'` reverse)
- `'reversed'`: existing reactions whose directionality should be flipped

The next step integrates this solution into a copy of the base model.

In [40]:
def _integrate_solution(template, model, gap_fill_solution):
    added_reactions = []
    for rxn_id, (lb, ub) in gap_fill_solution.items():
        template_reaction = template.reactions.get_by_id(rxn_id)
        model_reaction = template_reaction.to_reaction(model)
        model_reaction.lower_bound = lb
        model_reaction.upper_bound = ub
        _str = model_reaction.build_reaction_string(True)
        #print(f'{model.id} add {model_reaction.id}: {_str}')
        added_reactions.append(model_reaction)
    model.add_reactions(added_reactions)
    add_exchanges = MSBuilder.add_exchanges_to_model(model)

    return added_reactions, add_exchanges

model_gapfilled = model_base.copy()

from modelseedpy.core.msmodel import get_reaction_constraints_from_direction
from modelseedpy import MSBuilder
gap_sol = {}
for rxn_id, d in gapfill_res['new'].items():
    if rxn_id[:-1] in template_gramneg.reactions:
        gap_sol[rxn_id[:-1]] = get_reaction_constraints_from_direction(d)
print(gap_sol)
_integrate_solution(template_gramneg, model_gapfilled, gap_sol)

{'rxn12008_c': (-1000, 0), 'rxn08333_c': (0, 1000), 'rxn02898_c': (0, 1000), 'rxn05459_c': (0, 1000), 'rxn05514_c': (-1000, 0), 'rxn02160_c': (0, 1000), 'rxn00340_c': (0, 1000), 'rxn05039_c': (0, 1000), 'rxn03891_c': (0, 1000), 'rxn09997_c': (-1000, 0), 'rxn09042_c': (0, 1000), 'rxn03536_c': (0, 1000), 'rxn05573_c': (0, 1000), 'rxn00856_c': (0, 1000), 'rxn00003_c': (-1000, 0), 'rxn03537_c': (0, 1000), 'rxn12376_c': (-1000, 0), 'rxn04675_c': (0, 1000), 'rxn06023_c': (0, 1000), 'rxn13906_c': (-1000, 0), 'rxn01539_c': (-1000, 0), 'rxn10094_c': (0, 1000), 'rxn02155_c': (0, 1000), 'rxn03150_c': (0, 1000), 'rxn03538_c': (0, 1000), 'rxn09310_c': (0, 1000), 'rxn01213_c': (0, 1000), 'rxn02304_c': (0, 1000), 'rxn05618_c': (0, 1000), 'rxn03393_c': (0, 1000), 'rxn00599_c': (0, 1000), 'rxn09040_c': (0, 1000), 'rxn00551_c': (0, 1000), 'rxn00216_c': (0, 1000), 'rxn01790_c': (-1000, 0), 'rxn02897_c': (0, 1000), 'rxn05144_c': (0, 1000), 'rxn02831_c': (0, 1000), 'rxn02832_c': (-1000, 0), 'rxn07465_c': (

([<Reaction rxn12008_c0 at 0x7d298d61b430>,
  <Reaction rxn08333_c0 at 0x7d298d6182e0>,
  <Reaction rxn02898_c0 at 0x7d298d6186d0>,
  <Reaction rxn05459_c0 at 0x7d298d618700>,
  <Reaction rxn05514_c0 at 0x7d298d618a30>,
  <Reaction rxn02160_c0 at 0x7d298d618a00>,
  <Reaction rxn00340_c0 at 0x7d298d6190f0>,
  <Reaction rxn05039_c0 at 0x7d298d618f10>,
  <Reaction rxn03891_c0 at 0x7d298d618b80>,
  <Reaction rxn09997_c0 at 0x7d298d619750>,
  <Reaction rxn09042_c0 at 0x7d298d619d50>,
  <Reaction rxn03536_c0 at 0x7d298d61a170>,
  <Reaction rxn05573_c0 at 0x7d298d61a440>,
  <Reaction rxn00856_c0 at 0x7d298d619ae0>,
  <Reaction rxn00003_c0 at 0x7d298d61a7a0>,
  <Reaction rxn03537_c0 at 0x7d298d61a590>,
  <Reaction rxn12376_c0 at 0x7d298d61b0d0>,
  <Reaction rxn04675_c0 at 0x7d298d61b310>,
  <Reaction rxn06023_c0 at 0x7d298d61b250>,
  <Reaction rxn13906_c0 at 0x7d298d8b3970>,
  <Reaction rxn01539_c0 at 0x7d298d61b340>,
  <Reaction rxn10094_c0 at 0x7d298d8b3940>,
  <Reaction rxn02155_c0 at 0x7d2

`_integrate_solution` converts the gapfill solution into COBRApy reactions and adds them to a copy of the base model (`model_gapfilled`). Exchange reactions are also created for any new boundary metabolites. The gap solution dictionary maps template reaction IDs to flux bound tuples derived from the gapfill direction.

In [41]:
import math
def integrate_to_model_medium(media, model, prefix='EX_'):
    medium = {}
    for cpd, (lb, ub) in media.get_media_constraints().items():
        rxn_exchange = f'{prefix}{cpd}'
        if rxn_exchange in model.reactions:
            medium[rxn_exchange] = math.fabs(lb)
        else:
            print('not in model', cpd)
    return medium
model_gapfilled.medium = integrate_to_model_medium(media_glucose, model_gapfilled)
model_gapfilled.objective = 'bio1'
model_gapfilled.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
cpd00007_e0,EX_cpd00007_e0,8.613,0,0.00%
cpd00009_e0,EX_cpd00009_e0,0.3622,0,0.00%
cpd00013_e0,EX_cpd00013_e0,4.38,0,0.00%
cpd00027_e0,EX_cpd00027_e0,5,6,100.00%
cpd00030_e0,EX_cpd00030_e0,0.0008768,0,0.00%
cpd00034_e0,EX_cpd00034_e0,0.0008768,0,0.00%
cpd00048_e0,EX_cpd00048_e0,0.1131,0,0.00%
cpd00058_e0,EX_cpd00058_e0,0.0008768,0,0.00%
cpd00063_e0,EX_cpd00063_e0,0.0008768,0,0.00%
cpd00099_e0,EX_cpd00099_e0,0.0008768,0,0.00%


`integrate_to_model_medium` translates the `MSMedia` compound constraints into COBRApy exchange reaction bounds and applies them to the model. Setting the objective to `bio1` and calling `model.summary()` confirms that the gapfilled model predicts non-zero biomass production on glucose minimal medium.

In [43]:
model_gapfilled.medium = {}
model_gapfilled.objective = 'ATPM_c0'
sol = model_gapfilled.optimize()
print(sol)

<Solution 0.000 at 0x7d298d4fbcd0>


As a final sanity check, the model is run with an **empty medium** and ATPM as the objective. A near-zero flux confirms that the ATP correction step successfully eliminated thermodynamically infeasible energy-generating cycles.

The model is now complete and ready for downstream analyses such as flux balance analysis (FBA), gene essentiality screens, or media optimization.